# Hackathon project

The whole project, top to bottom: problem, data, baseline, model, evaluation, demo.
Run it with **Restart kernel and run all**. It must pass at every moment.

In [1]:
# Setup: pick up changes to src/ without restarting, and make `import src` work from notebooks/
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, "..")
import src

In [2]:
# One seed for everything random, so every run gives the same numbers
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("seed:", SEED)

seed: 42


## Problem
**Who it is for:** the owner of a small UK supplier, deciding whether to give a young company 30 days of credit.

**What goes in and what comes out:** a customer's company number goes in. FLAG, CLEAR or NOT ENOUGH EVIDENCE comes out, with the reason.

**Why it is hard:** after a company goes insolvent, the same people can start again in a new one (a "phoenix"). A matching name is not proof. A common name raises false alarms, and a new officer id on the register hides the real trail.

This project is a prototype and is not financial advice.

> **Where this notebook stands.** Every box below runs on the *tiny world*: 28 hand-made rows with invented names and known answers. That checks that the pipeline works. It is **not a result**. The real Companies House data comes with card #36, and the final numbers with card #45.

## Data
For now the data is the tiny world (`tests/tiny_world.py`). It comes as the same two tables the real data will: **mentions** (one row per company or person, as one record describes them) and **claims** (one row per "this person is a director of this company", with its dates).

In [3]:
import pandas as pd
from src import evidence as ev
from tests.tiny_world import tiny_world

# truth holds the known answers. It checks the pipeline, and it makes the stand-in probabilities
# in the Model section. No real method ever reads it.
mentions, claims, truth = tiny_world()
ev.check_rows(mentions, claims)    # stops here if a row breaks the shared schema
queries = truth["queries"]         # the five companies a supplier asks about

print(mentions["kind"].value_counts().to_dict(), "and", len(claims), "claims")
mentions.loc[mentions["kind"] == "company", ["company_number", "display_name", "incorporated_on", "insolvency_on"]]

{'company': 9, 'person': 9} and 10 claims


,company_number,display_name,incorporated_on,insolvency_on
0,TW000001,Harrow Build Ltd,2017-02-01,2021-03-15
1,TW000002,Harrow Build Services Ltd,2021-04-20,2022-09-01
2,TW000003,Northgate Roofing Ltd,2021-06-01,NaN
3,TW000004,Smith and Sons Joinery Ltd,2015-05-01,2020-10-01
4,TW000005,Elm Street Plumbing Ltd,2021-02-10,NaN
5,TW000006,Quiet Fields Landscaping Ltd,2021-09-01,NaN
6,TW000007,Brightwater Electrical Ltd,2021-05-15,NaN
7,TW000008,Lambert Electrical Ltd,2016-03-01,NaN
8,TW000009,Smith Brothers Builders Ltd,2023-01-10,2024-05-01


**The label and the split.** A company *failed* if it went into liquidation or administration within 24 months of being formed. Companies are split by formation date: train on Jan–Sep 2021, validate on Oct–Dec 2021, test on 2022.

In [4]:
from src import evaluate

companies = mentions[mentions["kind"] == "company"].set_index("company_number").loc[queries]
labels = pd.DataFrame({
    "company_id": queries,
    "formed_on": companies["incorporated_on"].to_numpy(),
    "failed": evaluate.label_failures(mentions).loc[queries].to_numpy(),
    "split": None,
}, columns=ev.LABEL_COLS)

# split() gives three tables: write each one's name next to its companies
for name, part in zip(["train", "validation", "test"], evaluate.split(companies)):
    labels.loc[labels["company_id"].isin(part.index), "split"] = name

assert dict(zip(labels["company_id"], labels["failed"])) == truth["failed"]   # the known answers
labels

,company_id,formed_on,failed,split
0,TW000002,2021-04-20,True,train
1,TW000003,2021-06-01,False,train
2,TW000005,2021-02-10,False,train
3,TW000006,2021-09-01,False,train
4,TW000007,2021-05-15,False,train


All five tiny companies were formed between January and September 2021, so they all land in train. The tiny world has no test set.

## Baseline
Three simple rules our method has to beat:
1. **flag_nothing:** never flag. Failures are rare, so this is right most of the time.
2. **same_name:** two directors with the same name are the same person. What someone does by hand.
3. **officer_links:** only the links Companies House already makes through its officer id. What a careful supplier sees on the register today.

A company is flagged when one of its directors is linked to a company that went insolvent in the three years before it was formed.

In [5]:
from src import baseline

baseline_decisions = baseline.run_all(queries, mentions, claims)
table = baseline_decisions.pivot(index="query", columns="method", values="decision")
table["really failed"] = labels.set_index("company_id")["failed"]
table

method,flag_nothing,officer_links,same_name,really failed
query,,,,
TW000002,CLEAR,CLEAR,FLAG,True
TW000003,CLEAR,CLEAR,FLAG,False
TW000005,CLEAR,CLEAR,FLAG,False
TW000006,CLEAR,CLEAR,CLEAR,False
TW000007,CLEAR,CLEAR,CLEAR,False


In [6]:
# Count each baseline's flags against what really happened.
# To do: use evaluate.score_methods() instead once it is on main (card #80).
flagged = baseline_decisions["decision"] == "FLAG"
failed = baseline_decisions["query"].map(labels.set_index("company_id")["failed"])
counts = pd.DataFrame({"flags": flagged, "right flags": flagged & failed,
                       "false flags": flagged & ~failed, "misses": failed & ~flagged})
counts.groupby(baseline_decisions["method"]).sum()

,flags,right flags,false flags,misses
method,,,,
flag_nothing,0,0,0,1
officer_links,0,0,0,1
same_name,3,1,2,0


**What the tiny world shows** (five companies and one real failure, so a check and not a result):
- `flag_nothing` and `officer_links` flag nothing, so both miss the phoenix. The register gave its director a new officer id, and the officer links can't see past that.
- `same_name` catches the phoenix, but it also flags two honest companies whose director shares a name with someone else.

Those two kinds of mistake, the missed phoenix and the false alarm on a shared name, are what our method is meant to avoid.

## Model
A company number goes in. For each of its directors we ask: is this the same person as someone who ran a company that went insolvent earlier? Each possible match gets a probability instead of a yes or no. From those we compute P(predecessor), then P(fail), then the decision.

**P(predecessor).** Each candidate match is a chance that a director is the same person as someone who ran an insolvent earlier company. The chance that at least one match is right is 1 − (1 − p₁)(1 − p₂)…, which assumes the matches are independent: whether one is right tells us nothing about another. That isn't quite true, because a director is only one person; Sunday's one-identity rule fixes it (ADR-004).

In [7]:
from src.decide import p_predecessor

# Two independent 50% matches: the chance both are wrong is 0.5 × 0.5, so at least one is right with 0.75
p_predecessor([0.5, 0.5])

0.75

**The decision.** A straight line turns P(predecessor) into P(fail). Three costs then say what to do: a missed case costs 10, a wrong flag 2, asking 1. That gives CLEAR at or below 10%, FLAG at or above 50%, and NOT ENOUGH EVIDENCE in between. A company with no director records is NOT ENOUGH EVIDENCE too.

> **Stand-ins.** The matcher that gives each pair of person records its probability is not built yet (cards #30, #31), and neither is the failure line (card #33). So the probabilities and the line below are **made up by hand** from the tiny world's known answers: 0.9 for a true same-person pair, 0.05 for a look-alike. This shows that the decision layer works. It says nothing about how good our method is.

In [8]:
from src import decide

# STAND-INS, made up by hand until the matcher (#31) and the failure line (#33) exist.
pairs = pd.DataFrame(
    [(a, b, 0.9, None, "stand-in", None) for a, b in truth["same_person"]]
    + [(a, b, 0.05, None, "stand-in", None) for a, b in truth["look_alikes"]],
    columns=ev.PAIR_COLS)
line = (0.05, 0.6)    # made up: 5% of companies fail without an insolvent predecessor, 60% with one
costs = (10, 2, 1)    # the team's choice: a missed case, a wrong flag, asking

rows = []
for query in queries:
    matches = decide.candidate_matches(query, mentions, claims, pairs)
    rows.append(decide.decide(query, matches, line, costs))
our_decisions = pd.DataFrame(rows, columns=ev.DECISION_COLS)

assert dict(zip(our_decisions["query"], our_decisions["decision"])) == truth["decision"]   # the known answers
our_decisions[["query", "decision", "p_predecessor", "p_fail"]]

,query,decision,p_predecessor,p_fail
0,TW000002,FLAG,0.90,0.5450
1,TW000003,CLEAR,0.05,0.0775
2,TW000005,CLEAR,0.05,0.0775
3,TW000006,NOT ENOUGH EVIDENCE,NaN,NaN
4,TW000007,CLEAR,0.00,0.0500


## Evaluation
_Ours next to the baselines on data the model never saw, and where it fails._

Not yet. So far only the baselines are counted, in the Baseline section above, and only on the tiny world: five companies, one of which failed. Our method joins the table once its probabilities come from the matcher instead of by hand (card #31). The numbers that count come from the 2022 test companies, run once at the end (card #45).

## Demo
One company number in, the result out. This is the tiny world's phoenix, with the stand-in probabilities from the Model section.

In [9]:
answer = our_decisions.set_index("query").loc["TW000002"]
print("(tiny world, stand-in probabilities: a pipeline check, not a result)")
print(answer["decision"])
print(answer["reason"])

(tiny world, stand-in probabilities: a pipeline check, not a result)
FLAG
TW000002 may share a director with TW000001, an earlier company that went insolvent (90% sure it's the same person): its 54% chance of failing within two years is high enough to flag.
